<a href="https://colab.research.google.com/github/Nidhi-Pawar/bert_sentiment_analysis_mini-project/blob/main/notebooks/BERT_LSTM_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project fine-tunes a pre-trained BERT model with an additional LSTM layer for the task of sentiment classification (positive/negative) using the SST-2 dataset.

In [ ]:
 !pip install torch torchvision torchaudio transformers scikit-learn datasets evaluate

In [ ]:
# Install libraries

import numpy as np
from datasets import load_dataset

from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModel

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import accuracy_score, f1_score


In [ ]:
raw_dataset = load_dataset("glue", "sst2")
raw_dataset

In [ ]:
train_ds = raw_dataset["train"]
train_ds[10]

In [ ]:
train_ds.features

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def tokenize_data(data):
  res = tokenizer(data["sentence"], truncation=True, padding=False, max_length=None)
  res['labels'] = data['label']
  return res

tokenized_datasets = raw_dataset.map(tokenize_data, batched=True, remove_columns=["sentence", "label"])

In [ ]:
tokenized_datasets

In [ ]:
# Dynamic Padding for train, validation and test datasets
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors = "pt")

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle = True,
    batch_size = 32,
    collate_fn = data_collator,
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    shuffle = True,
    batch_size = 32,
    collate_fn = data_collator,
)

test_dataloader = DataLoader(
    tokenized_datasets["test"],
    shuffle = True,
    batch_size = 32,
    collate_fn = data_collator,
)

In [ ]:
train_ds[5675]

In [ ]:
# Defining the BERT+LSTM model

class BERT_LSTM(nn.Module):
  def __init__(self, bert_model_name = checkpoint, hidden_dim=256, num_classes=2, dropout_rate=0.3):
    super(BERT_LSTM, self).__init__()
    self.bert = AutoModel.from_pretrained(bert_model_name)
    self.dropout1 = nn.Dropout(dropout_rate) # Add dropout after BERT for regularization

    #Bidirectional LSTM
    self.lstm = nn.LSTM(
        input_size = self.bert.config.hidden_size,
        hidden_size=hidden_dim,
        batch_first=True,
        bidirectional=True,
        dropout=dropout_rate if hidden_dim > 1 else 0)

    self.dropout2 = nn.Dropout(dropout_rate) # Additional dropout before classification

    self.classifier = nn.Linear(hidden_dim*2, num_classes)


  def forward(self, input_ids, attention_mask):
    # Passing inputs through BERT
    outputs = self.bert(input_ids = input_ids, attention_mask = attention_mask)
    sequence_output = outputs[0]

    # Apply dropout after BERT
    sequence_output = self.dropout1(sequence_output)

    # Pass BERT output through LSTM head
    lstm_out, _ = self.lstm(sequence_output)

    # Get output from last token and apply dropout
    final_hidden = self.dropout2(lstm_out[:, -1, :])

    # Get output from last token
    output = self.classifier(final_hidden)

    return output

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
torch.cuda.is_available()

In [ ]:
# Setting up the training parameters

# Initializing the model, optimizer and loss function
model = BERT_LSTM()
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

# Moving the model to available GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print
model.to(device)

In [ ]:
from tqdm.notebook import tqdm
from torch.amp import autocast, GradScaler

# Training Loop
scaler = GradScaler()
def train(model, train_dataloader, optimizer, loss_fn, device):
  model.train()
  total_loss=0
  all_preds=[]
  all_labels=[]

  for batch in tqdm(train_dataloader):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    # Zero out the gradients
    optimizer.zero_grad()

    with autocast(device_type='cuda'):

      # Perform forward pass and compute loss
      outputs = model(input_ids, attention_mask)
      logits = outputs

      preds = torch.argmax(logits, dim=1)
      # preds = (logits > 0.5).long()
      loss = loss_fn(logits.squeeze(), labels)

      # Perform Backpropogation
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      # Track metrics
      total_loss += loss.item()

      # labels = torch.argmax(labels, dim=1)

      all_preds.extend(preds.cpu().detach().numpy())
      all_labels.extend(labels.cpu().detach().numpy())

  all_preds = np.array(all_preds, dtype=int)
  all_labels = np.array(all_labels, dtype=int)
  avg_loss = total_loss / len(train_dataloader)
  accuracy = accuracy_score(all_labels, all_preds)
  f1 = f1_score(all_labels, all_preds, average='weighted')

  return avg_loss, accuracy, f1


In [ ]:
# Validation Loop to track overfitting
def evaluate(model, val_dataloader, loss_fn, device):
  model.eval()
  total_loss = 0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch in tqdm(val_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with autocast(device_type='cuda'):
          outputs = model(input_ids, attention_mask)
          logits = outputs
          preds = torch.argmax(logits, dim=1)
          # loss = loss_fn(logits, labels)

          # preds = (logits > 0.5).long()
          loss = loss_fn(logits.squeeze(), labels)

          total_loss += loss.item()

          # labels = torch.argmax(labels, dim=1)
          all_preds.extend(preds.cpu().detach().numpy())
          all_labels.extend(labels.cpu().detach().numpy())

    all_preds = np.array(all_preds, dtype=int)
    all_labels = np.array(all_labels, dtype=int)

    avg_loss = total_loss / len(val_dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, accuracy, f1


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.005, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model, path='best_model.pt'):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'Early stopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, path):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), path)
        self.val_loss_min = val_loss


In [ ]:
# Setting up the training pipeline

def train_model(model, train_dataloader, val_dataloader, device, num_epochs=3, lr=2e-5):
    # Setup the parameters
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()

    # Early stopping
    early_stopping = EarlyStopping(patience=3, verbose=True)

    # Learning rate scheduler - updates based on validation loss
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    # Training and validation history
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }

    # Main training loop
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        # Training phase
        train_loss, train_acc, train_f1 = train(model, train_dataloader, optimizer, loss_fn, device)
        print(f"Train Loss: {train_loss:.3f} | Train Accuracy: {train_acc:.3f} | Train F1: {train_f1:.3f}")

        # Validation phase
        val_loss, val_acc, val_f1 = evaluate(model, val_dataloader, loss_fn, device)
        print(f"Validation Loss: {val_loss:.3f} | Validation Accuracy: {val_acc:.3f} | Validation F1: {val_f1:.3f}")

        # Update learning rate based on validation loss
        scheduler.step(val_loss)

        # Check early stopping
        early_stopping(val_loss, model)

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

    # Load best model
    # model.load_state_dict(torch.load('best_model.pt'))

    return model, history


In [ ]:
# torch.save(model.state_dict(), "bert_lstm_sentiment.pth")
# from google.colab import files
# files.download("bert_lstm_sentiment.pth")

In [ ]:
# Set training parameters
num_epochs = 3
learning_rate = 2e-5

# Call the train_model function
trained_model, history = train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    device=device,
    num_epochs=num_epochs,
    lr=learning_rate
)

# Visualize the training history
import matplotlib.pyplot as plt

# Setup figure and subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Plot Loss
epochs = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

# Plot Accuracy
ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

# Plot F1 Score
ax3.plot(epochs, history['train_f1'], 'b-', label='Training F1')
ax3.plot(epochs, history['val_f1'], 'r-', label='Validation F1')
ax3.set_title('Training & Validation F1 Score')
ax3.set_xlabel('Epochs')
ax3.set_ylabel('F1 Score')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
for i in range(num_epochs):
  print(f"Epoch {i+1}/{num_epochs}")
  print(f"Train loss: {history['train_loss'][i]}")
  print(f"Train accuracy: {history['train_acc'][i]}")
  print(f"Train F1 score: {history['train_f1'][i]}")
  print(f"Validation loss: {history['val_loss'][i]}")
  print(f"Validation accuracy: {history['val_acc'][i]}")
  print(f"Validation F1 score: {history['val_f1'][i]}")
  print('\n')

In [ ]:
# Since the sst2 test data does not specify label, we can only make predictions, and cannot calculate the metrics
all_preds = []
test_ds = raw_dataset["test"]

model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())


In [ ]:
for i in range (5):
  rndm_idx = np.random.randint(0, len(test_ds), 1)
  print(f"Test index number:{rndm_idx}")
  print(f"Sentence: {test_ds[rndm_idx]['sentence']}")
  print(f"Prediction:{all_preds[rndm_idx.item()]}")
  print('\n')
